# Notebook 15 — Report Generation

## Purpose
I compile key summary statistics into tables suitable for the report,
and print a structured summary of all findings.

## Outputs
`reports/tables/executive_summary.csv`, `paper_or_report/tables/executive_summary.csv`


In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

sys.path.insert(0, str(Path('..').resolve()))
from src.config import load_config
from src.paths import Paths
from src.utils import save_table, load_metrics

cfg   = load_config()
paths = Paths(cfg)

master = pd.read_parquet(paths.processed / cfg['data']['master_file'])
rfm    = pd.read_parquet(paths.processed / cfg['data']['rfm_file'])
advanced = load_metrics(paths.models / 'advanced_results.json')
baseline = load_metrics(paths.models / 'baseline_results.json')
print("All artifacts loaded.")


All artifacts loaded.


In [2]:
# --- Top-level KPIs ---
kpis = {
    'total_orders':          len(master),
    'total_revenue_brl':     round(master['order_value'].sum(), 2),
    'avg_order_value_brl':   round(master['order_value'].mean(), 2),
    'avg_review_score':      round(master['review_score'].mean(), 3),
    'late_delivery_pct':     round(master['is_late'].mean() * 100, 2),
    'unique_customers':      master['customer_unique_id'].nunique(),
    'date_from':             str(master['order_purchase_timestamp'].min().date()),
    'date_to':               str(master['order_purchase_timestamp'].max().date()),
    'report_generated':      datetime.now().strftime('%Y-%m-%d %H:%M'),
}

print("=== Project KPIs ===")
for k, v in kpis.items():
    print(f"  {k:<35} {v}")


=== Project KPIs ===
  total_orders                        96478
  total_revenue_brl                   13221498.11
  avg_order_value_brl                 137.04
  avg_review_score                    4.156
  late_delivery_pct                   6.77
  unique_customers                    93358
  date_from                           2016-09-15
  date_to                             2018-08-29
  report_generated                    2026-06-13 03:00


In [3]:
# --- Segmentation summary ---
seg_summary = rfm.groupby('segment').agg(
    n_customers=('customer_unique_id', 'count'),
    avg_recency_days=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary_brl=('monetary', 'mean'),
    total_revenue_brl=('monetary', 'sum'),
).round(2).reset_index()
seg_summary['revenue_pct'] = (
    seg_summary['total_revenue_brl'] / seg_summary['total_revenue_brl'].sum() * 100
).round(2)
seg_summary = seg_summary.sort_values('total_revenue_brl', ascending=False)
print("=== Segmentation Summary ===")
display(seg_summary)
save_table(seg_summary, 'segmentation_summary',
           reports_dir=str(paths.reports_tabs),
           paper_dir=str(paths.paper_tabs))


=== Segmentation Summary ===


,segment,n_customers,avg_recency_days,avg_frequency,avg_monetary_brl,total_revenue_brl,revenue_pct
1,At Risk,15490,380.98,1.05,201.93,3127833.08,23.66
9,Other,14684,142.45,1.02,143.03,2100187.63,15.88
3,Champions,6474,90.89,1.17,274.16,1774913.54,13.42
2,Cant Lose Them,5874,436.15,1.06,240.29,1411442.64,10.68
11,Promising,6679,134.06,1.00,204.96,1368948.91,10.35
7,Need Attention,5762,155.79,1.00,190.51,1097741.57,8.30
6,Loyal,8153,183.63,1.08,127.40,1038686.00,7.86
4,Hibernating,10566,337.37,1.00,53.76,568045.66,4.30
10,Potential Loyalist,8222,76.66,1.00,40.53,333220.25,2.52
0,About to Sleep,3702,256.53,1.00,43.07,159438.22,1.21


  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\reports\tables\segmentation_summary.csv
  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\paper_or_report\tables\segmentation_summary.csv


[WindowsPath('C:/Users/Peter/Documents/projects/Jobberman_projects/double_Integral/ecommerce_customer_intelligence/reports/tables/segmentation_summary.csv'),
 WindowsPath('C:/Users/Peter/Documents/projects/Jobberman_projects/double_Integral/ecommerce_customer_intelligence/paper_or_report/tables/segmentation_summary.csv')]

In [4]:
# --- Model performance summary ---
model_perf = pd.DataFrame([
    {'Task': 'Review Score (5-class)',
     'Model': 'Dummy (baseline)',
     'Accuracy': round(baseline['task_review_score']['dummy']['point'], 4),
     'Macro_F1': 'N/A'},
    {'Task': 'Review Score (5-class)',
     'Model': 'Logistic Regression',
     'Accuracy': round(baseline['task_review_score']['logreg']['point'], 4),
     'Macro_F1': 'N/A'},
    {'Task': 'Review Score (5-class)',
     'Model': 'Random Forest',
     'Accuracy': round(advanced['task_review_score']['random_forest']['point'], 4),
     'Macro_F1': round(advanced['task_review_score']['random_forest']['macro_f1'], 4)},
    {'Task': 'Review Score (5-class)',
     'Model': 'XGBoost',
     'Accuracy': round(advanced['task_review_score']['xgboost']['point'], 4),
     'Macro_F1': round(advanced['task_review_score']['xgboost']['macro_f1'], 4)},
    {'Task': 'Late Delivery (binary)',
     'Model': 'Random Forest',
     'Accuracy': round(advanced['task_late_delivery']['random_forest']['accuracy'], 4),
     'Macro_F1': round(advanced['task_late_delivery']['random_forest']['macro_f1'], 4)},
])
print("=== Model Performance Summary ===")
display(model_perf)
save_table(model_perf, 'model_performance_summary',
           reports_dir=str(paths.reports_tabs),
           paper_dir=str(paths.paper_tabs))
print("\nNotebook 15 complete. All summary tables saved.")


=== Model Performance Summary ===


,Task,Model,Accuracy,Macro_F1
0,Review Score (5-class),Dummy (baseline),0.5933,N/A
1,Review Score (5-class),Logistic Regression,0.3065,N/A
2,Review Score (5-class),Random Forest,0.5493,0.2924
3,Review Score (5-class),XGBoost,0.6238,0.2551
4,Late Delivery (binary),Random Forest,0.9226,0.4799


  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\reports\tables\model_performance_summary.csv
  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\paper_or_report\tables\model_performance_summary.csv

Notebook 15 complete. All summary tables saved.
